# Tokenization — How Machines Read Text

In [9]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q tiktoken transformers

## How does a machine read text?

When you read the sentence "The cat sat on the mat", you understand it instantly. You know what each word means, how they relate to each other, and what the whole sentence is saying.

A machine cannot do that directly. Machines only understand numbers — not letters, not words, not sentences. So before any AI model can process text, the text needs to be converted into numbers.

That conversion process is called **tokenization**.

**A simple analogy:** Think about how a child learns to read. They do not start by reading whole sentences. They start with individual sounds — "c", "a", "t" — and learn to combine them into words. Tokenization works in a similar way. It breaks text into smaller pieces first, then maps each piece to a number.

Those pieces are called **tokens**, and the numbers they map to are called **token IDs**.

## The three ways to tokenize text

There is more than one way to break text into pieces. Here are the three approaches, and why the first two have problems.

**1. Word Tokenization** — split on spaces

"I love Biryani" → ["I", "love", "pizza"]

Simple, but has a big problem. "run", "runs", "running", and "runner" are all treated as completely different words. The vocabulary gets enormous. And any word the model has never seen — a rare word, a typo, a proper noun — is completely unrecognizable.

**2. Character Tokenization** — split letter by letter

"cat" → ["c", "a", "t"]

This keeps the vocabulary tiny (just the alphabet), but the sequences become very long. A single paragraph turns into hundreds of individual characters. The model has to work much harder to find meaning in them.

**3. Subword Tokenization** — the best of both (what modern models use)

Common words stay whole: "cat" → ["cat"]
Rare or complex words split into recognizable parts: "unbelievable" → ["un", "believe", "able"]

This keeps the vocabulary manageable, handles words the model has never seen, and preserves meaning better than splitting by character. GPT, BERT, LLaMA, and almost every modern model uses this approach.

In [10]:
text = "The weather is unbelievably beautiful today"

# Word tokenization — split on spaces
word_tokens = text.split()
print("Word tokens:")
print(word_tokens)
print(f"Total: {len(word_tokens)} tokens\n")

# Character tokenization — split into individual characters
char_tokens = list(text)
print("Character tokens (first 20):")
print(char_tokens[:20])
print(f"Total: {len(char_tokens)} tokens")
print("\nNotice how character tokenization produces a much longer sequence.")

Word tokens:
['The', 'weather', 'is', 'unbelievably', 'beautiful', 'today']
Total: 6 tokens

Character tokens (first 20):
['T', 'h', 'e', ' ', 'w', 'e', 'a', 't', 'h', 'e', 'r', ' ', 'i', 's', ' ', 'u', 'n', 'b', 'e', 'l']
Total: 43 tokens

Notice how character tokenization produces a much longer sequence.


## Seeing it in action — GPT's tokenizer (tiktoken)

The `tiktoken` library is the tokenizer used by OpenAI's GPT models. It uses subword tokenization, so common words stay as single tokens while rare or complex words get split up.

The encoding `cl100k_base` is the one used by GPT-4 and GPT-3.5-turbo. Let's see how it handles our sentence.

In [11]:
import tiktoken

# cl100k_base is the encoding used by GPT-4 and GPT-3.5-turbo
encoder = tiktoken.get_encoding("cl100k_base")

text = "The weather is unbelievably beautiful today"

tokens = encoder.encode(text)
print(f"Text: {text}")
print(f"Token IDs: {tokens}")
print(f"Total tokens: {len(tokens)}\n")

print("Each token ID mapped back to the text it represents:")
for token_id in tokens:
    print(f"  {token_id:>8}  →  '{encoder.decode([token_id])}'")

Text: The weather is unbelievably beautiful today
Token IDs: [791, 9282, 374, 40037, 89234, 6366, 3432]
Total tokens: 7

Each token ID mapped back to the text it represents:
       791  →  'The'
      9282  →  ' weather'
       374  →  ' is'
     40037  →  ' unbelie'
     89234  →  'vably'
      6366  →  ' beautiful'
      3432  →  ' today'


## Comparing tokenizers — GPT vs BERT

Different models use different tokenizers, each trained differently with different vocabularies. The same text can produce different tokens depending on which model you are using.

GPT models use **BPE (Byte Pair Encoding)**. BERT uses **WordPiece**. They produce different token splits, different IDs, and sometimes a different number of tokens for the same input.

Let's see how BERT handles the same sentence.

In [12]:
from transformers import AutoTokenizer

# BERT's tokenizer — 'uncased' means it lowercases everything
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "The weather is unbelievably beautiful today"

tokens = bert_tokenizer.tokenize(text)
ids    = bert_tokenizer.convert_tokens_to_ids(tokens)

print(f"Text: {text}\n")
print(f"BERT tokens: {tokens}")
print(f"Token IDs:   {ids}")
print()
print("Two things to notice:")
print("  1. BERT lowercases everything — that is what 'uncased' means.")
print("  2. BERT uses ## to mark continuation pieces.")
print("     '##ly' means 'ly' is attached to the previous token, not a new word.")

Text: The weather is unbelievably beautiful today

BERT tokens: ['the', 'weather', 'is', 'un', '##bel', '##ie', '##va', '##bly', 'beautiful', 'today']
Token IDs:   [1996, 4633, 2003, 4895, 8671, 2666, 3567, 6321, 3376, 2651]

Two things to notice:
  1. BERT lowercases everything — that is what 'uncased' means.
  2. BERT uses ## to mark continuation pieces.
     '##ly' means 'ly' is attached to the previous token, not a new word.


## Special tokens — the ones you never typed

When you use a real model, the tokenizer quietly adds extra tokens you did not write. These are called **special tokens**, and they carry instructions to the model about the structure of the input.

Common ones you will see:

- `[CLS]` — added at the very start by BERT. The model uses this token's final output for classification tasks.
- `[SEP]` — added at the end, and between two sentences. Tells the model where one piece of text ends and another begins.
- `[PAD]` — used to fill shorter sequences so that all sequences in a batch are the same length.
- `<|endoftext|>` — used by GPT models to mark the end of a document.

You will encounter these every time you work with HuggingFace models. Here is how to see them.

In [13]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "I love coffee"

encoded = bert_tokenizer(
    text,
    add_special_tokens=True,
    padding="max_length",
    max_length=10,
    return_attention_mask=True,
)

tokens = bert_tokenizer.convert_ids_to_tokens(encoded["input_ids"])

print(f"Input text : '{text}'\n")
print(f"Tokens     : {tokens}")
print(f"Token IDs  : {encoded['input_ids']}")
print(f"Attn mask  : {encoded['attention_mask']}")
print()
print("The attention mask tells the model which tokens are real (1)")
print("and which are just padding to fill the sequence length (0).")
print()
print(f"Special token IDs — [CLS]: {bert_tokenizer.cls_token_id},  [SEP]: {bert_tokenizer.sep_token_id},  [PAD]: {bert_tokenizer.pad_token_id}")

Input text : 'I love coffee'

Tokens     : ['[CLS]', 'i', 'love', 'coffee', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Token IDs  : [101, 1045, 2293, 4157, 102, 0, 0, 0, 0, 0]
Attn mask  : [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

The attention mask tells the model which tokens are real (1)
and which are just padding to fill the sequence length (0).

Special token IDs — [CLS]: 101,  [SEP]: 102,  [PAD]: 0


## Counting tokens — why this matters in practice

Every time you send text to a language model API, you are charged per token. The model also has a maximum number of tokens it can process in one request — this is called the **context window**.

GPT-5.5 supports up to 1 Million tokens (Input + Output). That sounds like a lot, but a single page of text is roughly 500–700 tokens. A long PDF can easily exceed the limit without you realising it.

Knowing how to count tokens before sending a request lets you:
- Estimate cost before running something expensive at scale
- Avoid hitting the context window limit on long documents
- Make smarter decisions about how to split or summarize text

In [14]:
import tiktoken

encoder = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoder.encode(text))

# A typical API request has a system prompt and a user message
system_prompt = """You are a helpful assistant. Answer questions clearly and concisely.
If you are unsure about something, say so rather than guessing."""

user_message = "Can you explain how transformer models work in simple terms?"

system_tokens = count_tokens(system_prompt)
user_tokens   = count_tokens(user_message)
total         = system_tokens + user_tokens

print(f"System prompt : {system_tokens:>4} tokens")
print(f"User message  : {user_tokens:>4} tokens")
print(f"Total input   : {total:>4} tokens")

# GPT-4o approximate pricing (mid-2024): $5 per 1M input tokens
cost_per_million = 5.0
estimated_cost = (total / 1_000_000) * cost_per_million
print(f"\nEstimated cost: ${estimated_cost:.6f}")
print("(Tiny for one call — but adds up fast when processing thousands of documents)")

System prompt :   27 tokens
User message  :   11 tokens
Total input   :   38 tokens

Estimated cost: $0.000190
(Tiny for one call — but adds up fast when processing thousands of documents)


## Not all languages are equal — multilingual tokenization

Most tokenizers were trained primarily on English text. This makes English very efficient — typically around 1 token per word.

Other languages, especially those with non-Latin scripts like Hindi or Tamil, can require 2–4 tokens per character. This means:

- A document in Tamil costs more tokens than the same content in English
- It hits the context window limit faster
- The model may have seen far less training data in that language, which can affect quality

Let's compare the same sentence across three languages.

In [15]:
import tiktoken

encoder = tiktoken.get_encoding("cl100k_base")

sentences = {
    "English" : "The weather is beautiful today",
    "Hindi"   : "आज मौसम बहुत सुंदर है",
    "Tamil"   : "இன்று வானிலை மிகவும் அழகாக உள்ளது",
}

print(f"{'Language':<10}  {'Tokens':>6}  Text")
print("-" * 60)
for lang, text in sentences.items():
    n = len(encoder.encode(text))
    print(f"{lang:<10}  {n:>6}  {text}")

print("\nThe same meaning takes significantly more tokens in non-Latin scripts.")
print("This is a real concern when building multilingual applications.")

Language    Tokens  Text
------------------------------------------------------------
English          5  The weather is beautiful today
Hindi           23  आज मौसम बहुत सुंदर है
Tamil           46  இன்று வானிலை மிகவும் அழகாக உள்ளது

The same meaning takes significantly more tokens in non-Latin scripts.
This is a real concern when building multilingual applications.


## Key takeaways

- Tokenization converts text into numbers before any model can process it.
- Modern models use **subword tokenization** — common words stay whole, rare words split into recognizable parts.
- Different models use different tokenizers. The same text produces different tokens depending on the model.
- Tokenizers add **special tokens** automatically — [CLS], [SEP], padding — that carry structural information to the model.
- Token count determines **API cost** and whether your text fits in the **context window**.
- Non-English languages are generally less token-efficient than English.

---

Next up: **Embeddings** — once text becomes token IDs, those IDs get mapped to dense vectors that actually carry meaning. That is where semantic understanding begins.